In [3]:
import pandas as pd

# Read the CSV files
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")
sample_submission_df = pd.read_csv("/content/sample_submission.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample Submission shape:", sample_submission_df.shape)

print("\nTrain columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())
print("Sample Submission columns:", sample_submission_df.columns.tolist())

Train shape: (2000, 8)
Test shape: (500, 7)
Sample Submission shape: (500, 2)

Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']
Sample Submission columns: ['ID', 'Prediction']


In [4]:
!pip -q uninstall -y transformers peft datasets accelerate tokenizers

In [5]:
!pip -q install \
transformers==4.51.3 \
datasets==3.6.0 \
accelerate==1.7.0 \
peft==0.15.2 \
tokenizers==0.21.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
import torch
import torch.nn.functional as F
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)

In [7]:
train_df = pd.read_csv("/content/train.csv")

In [8]:
label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

In [9]:
dataset = Dataset.from_pandas(train_df.iloc[:32])

In [10]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
def preprocess(example):

    first = [example["prompt"]]*5

    second = [
        example["A"],
        example["B"],
        example["C"],
        example["D"],
        example["E"]
    ]

    enc = tokenizer(
        first,
        second,
        padding="max_length",
        truncation=True,
        max_length=64,
    )

    enc["label"] = label_map[example["answer"]]

    return enc

In [12]:
dataset = dataset.map(preprocess)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [13]:
dataset.set_format(type=None)

In [14]:
import torch

def mc_data_collator(features):
    batch = {}

    batch["input_ids"] = torch.tensor(
        [f["input_ids"] for f in features],
        dtype=torch.long,
    )

    batch["attention_mask"] = torch.tensor(
        [f["attention_mask"] for f in features],
        dtype=torch.long,
    )

    batch["labels"] = torch.tensor(
        [f["label"] for f in features],
        dtype=torch.long,
    )

    return batch

In [15]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(model, config)

In [17]:
args = TrainingArguments(
    output_dir="./out",

    per_device_train_batch_size=4,

    gradient_accumulation_steps=1,

    max_steps=4,

    save_strategy="no",

    logging_steps=1,

    report_to="none",

    remove_unused_columns=False,
)

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    processing_class=tokenizer,
    data_collator=mc_data_collator,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [19]:
!pip uninstall -y torchvision
import torch
print(torch.__version__)

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
2.11.0+cu128


In [20]:
trainer.train()

Step,Training Loss
1,1.657800
2,1.682900
3,1.668600
4,1.664200


TrainOutput(global_step=4, training_loss=1.6683713495731354, metrics={'train_runtime': 1.8172, 'train_samples_per_second': 8.805, 'train_steps_per_second': 2.201, 'total_flos': 2640170250240.0, 'train_loss': 1.6683713495731354, 'epoch': 0.5})

In [36]:
# qn 10 below

In [21]:
model.eval()

row = train_df.iloc[0]

first = [row["prompt"]]*5

second = [
    row["A"],
    row["B"],
    row["C"],
    row["D"],
    row["E"]
]

enc = tokenizer(
    first,
    second,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

device = next(model.parameters()).device

inputs = {
    "input_ids": enc["input_ids"].unsqueeze(0).to(device),
    "attention_mask": enc["attention_mask"].unsqueeze(0).to(device),
}

if "token_type_ids" in enc:
    inputs["token_type_ids"] = enc["token_type_ids"].unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(**inputs).logits

probs = F.softmax(logits, dim=-1)

print("A:", probs[0,0].item())
print("B:", probs[0,1].item())
print("C:", probs[0,2].item())
print("D:", probs[0,3].item())
print("E:", probs[0,4].item())

print("\nAnswer =", round(probs[0,4].item(),4))

A: 0.20033417642116547
B: 0.19954410195350647
C: 0.1997104436159134
D: 0.19989286363124847
E: 0.20051845908164978

Answer = 0.2005


In [22]:
# Install (run once in Colab)
!pip -q install transformers==4.51.3 peft==0.15.2 accelerate==1.7.0

In [23]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, TaskType, get_peft_model

# Load the model
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable percentage: {100 * trainable_params / total_params:.4f}%")

# Optional detailed summary
model.print_trainable_parameters()

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 295,681
Total parameters: 109,778,690
Trainable percentage: 0.2693%
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [24]:
#qn1

# Label encoding mapping
label_mapping = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

# Create a new numeric label column
train_df["label"] = train_df["answer"].map(label_mapping)

# Display the answer and its encoded label for row index 150
print("Original Answer:", train_df.loc[150, "answer"])
print("Encoded Label:", train_df.loc[150, "label"])

Original Answer: C
Encoded Label: 2


In [25]:
#qn2

# Create the formatted input for Option B at row index 0
option_b_input = (
    str(train_df.loc[0, "prompt"]) +
    " [SEP] " +
    str(train_df.loc[0, "B"])
)

# Print the formatted string (optional)
print(option_b_input)

# Print its character length
print("Character Length:", len(option_b_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Character Length: 407


In [26]:
#qn 3
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Get prompt
prompt = str(train_df.loc[0, "prompt"])

# Create the five prompt-option pairs
choices = [
    str(train_df.loc[0, "A"]),
    str(train_df.loc[0, "B"]),
    str(train_df.loc[0, "C"]),
    str(train_df.loc[0, "D"]),
    str(train_df.loc[0, "E"]),
]

formatted_inputs = [prompt + " [SEP] " + choice for choice in choices]

# Tokenize
encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape for a multiple-choice model
input_ids = encoding["input_ids"].unsqueeze(0)

print("Shape:", input_ids.shape)

Shape: torch.Size([1, 5, 128])


In [27]:
# qn4

from transformers import AutoTokenizer
import torch

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

all_input_ids = []

# Tokenize the first 16 rows
for i in range(16):
    prompt = str(train_df.loc[i, "prompt"])

    choices = [
        str(train_df.loc[i, "A"]),
        str(train_df.loc[i, "B"]),
        str(train_df.loc[i, "C"]),
        str(train_df.loc[i, "D"]),
        str(train_df.loc[i, "E"])
    ]

    formatted_inputs = [prompt + " [SEP] " + choice for choice in choices]

    encoding = tokenizer(
        formatted_inputs,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    all_input_ids.append(encoding["input_ids"])

# Stack into a batch
input_ids = torch.stack(all_input_ids)

print("Shape:", input_ids.shape)

# Total token positions
total_positions = input_ids.numel()
print("Total token positions:", total_positions)

Shape: torch.Size([16, 5, 128])
Total token positions: 10240


In [29]:
#qn5

from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Row index 0
prompt = str(train_df.loc[0, "prompt"])
choices = [
    str(train_df.loc[0, "A"]),
    str(train_df.loc[0, "B"]),
    str(train_df.loc[0, "C"]),
    str(train_df.loc[0, "D"]),
    str(train_df.loc[0, "E"])
]

# Tokenize prompt-choice pairs
encoding = tokenizer(
    [prompt] * 5,
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape for Multiple Choice model
inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0),
}

if "token_type_ids" in encoding:
    inputs["token_type_ids"] = encoding["token_type_ids"].unsqueeze(0)

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# Logits
print("Logits shape:", outputs.logits.shape)
print(outputs.logits)

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Logits shape: torch.Size([1, 5])
tensor([[-0.2981, -0.2974, -0.3016, -0.2982, -0.2967]])


In [31]:
# qn 6

from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Encode labels if not already done
label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train_df["label"] = train_df["answer"].map(label_mapping)

# Row index 0
prompt = str(train_df.loc[0, "prompt"])
choices = [
    str(train_df.loc[0, "A"]),
    str(train_df.loc[0, "B"]),
    str(train_df.loc[0, "C"]),
    str(train_df.loc[0, "D"]),
    str(train_df.loc[0, "E"])
]

# Tokenize
encoding = tokenizer(
    [prompt] * 5,
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Prepare inputs
inputs = {
    "input_ids": encoding["input_ids"].unsqueeze(0),
    "attention_mask": encoding["attention_mask"].unsqueeze(0),
    "labels": torch.tensor([train_df.loc[0, "label"]])
}

if "token_type_ids" in encoding:
    inputs["token_type_ids"] = encoding["token_type_ids"].unsqueeze(0)

# Forward pass with label
outputs = model(**inputs)

# Loss
print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)
print("Number of dimensions:", outputs.loss.dim())

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loss: tensor(1.6090, grad_fn=<NllLossBackward0>)
Loss shape: torch.Size([])
Number of dimensions: 0


In [33]:
# qn7
# Install PEFT if needed
!pip -q install peft

from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

# Load the model
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Trainable parameters:", trainable_params)

# Optional: Detailed summary
model.print_trainable_parameters()

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [34]:
# qn8

from datasets import Dataset
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Encode labels
label_mapping = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
train_df["label"] = train_df["answer"].map(label_mapping)

# Take the first 100 rows
dataset = Dataset.from_pandas(train_df.iloc[:100])

# Preprocessing function
def preprocess(example):
    prompt = str(example["prompt"])
    choices = [
        str(example["A"]),
        str(example["B"]),
        str(example["C"]),
        str(example["D"]),
        str(example["E"])
    ]

    encoding = tokenizer(
        [prompt] * 5,
        choices,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": example["label"]
    }

# Apply preprocessing
tokenized_dataset = dataset.map(preprocess)

# Check the first item
print("input_ids shape:", (
    len(tokenized_dataset[0]["input_ids"]),
    len(tokenized_dataset[0]["input_ids"][0])
))
print("Number of tokenized choices:", len(tokenized_dataset[0]["input_ids"]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

input_ids shape: (5, 128)
Number of tokenized choices: 5
